### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

### Load Data

In [2]:
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"

model_df = pd.read_csv(PROCESSED_DIR / "modeling_dataset.csv")
pred_df = pd.read_csv(PROCESSED_DIR / "model_predictions.csv")
feature_importance = pd.read_csv(PROCESSED_DIR / "feature_importance.csv")

model_df["week"] = pd.to_datetime(model_df["week"])
pred_df["week"] = pd.to_datetime(pred_df["week"])

model_df.head()

,state,week,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,year,month,week_of_year,quarter,aqi_mean_lag1,aqi_mean_lag2,aqi_max_lag1,aqi_max_lag2,aqi_p90_lag1,aqi_p90_lag2,aqi_mean_rolling_3
0,AK,2021-01-16,59.71,36.651163,75.8,163,43,2021,1,2,1,44.414634,74.692308,174.0,189.0,116.0,156.8,51.919368
1,AK,2021-01-23,46.43,38.000000,87.0,116,41,2021,1,3,1,36.651163,44.414634,163.0,174.0,75.8,116.0,39.688599
2,AK,2021-01-30,41.57,45.975610,86.0,163,41,2021,1,4,1,38.000000,36.651163,116.0,163.0,87.0,75.8,40.208924
3,AK,2021-02-06,39.86,38.045455,64.4,93,44,2021,2,5,1,45.975610,38.000000,163.0,116.0,86.0,87.0,40.673688
4,AK,2021-02-13,33.43,46.475000,69.3,131,40,2021,2,6,1,38.045455,45.975610,93.0,163.0,64.4,86.0,43.498688


### Title and Summary

# Air Quality and Respiratory Admissions Dashboard

This dashboard summarizes the relationship between weekly air quality conditions and respiratory-related hospital admissions across U.S. states.

### What to look for
- How respiratory admissions change over time
- How well the predictive models perform
- Whether higher AQI tends to align with higher hospital burden
- Which AQI and time-based features are most important for prediction

The Random Forest model performed substantially better than Linear Regression, suggesting that the relationship between AQI and respiratory admissions is nonlinear.

### Create the widgets

In [3]:
state_dropdown = widgets.Dropdown(
    options=sorted(pred_df["state"].dropna().unique()),
    description="State:",
    value=sorted(pred_df["state"].dropna().unique())[0]
)

model_dropdown = widgets.Dropdown(
    options=["pred_rf", "pred_lr"],
    description="Model:",
    value="pred_rf"
)

aqi_dropdown = widgets.Dropdown(
    options=["aqi_mean", "aqi_p90", "aqi_max"],
    description="AQI Metric:",
    value="aqi_mean"
)

### Main dashboard function

In [4]:
def update_dashboard(state, model_col, aqi_metric):
    clear_output(wait=True)

    display(Markdown("""
# Air Quality and Respiratory Admissions Dashboard

This dashboard summarizes the relationship between weekly air quality conditions and respiratory-related hospital admissions across U.S. states.

### What to look for
- How respiratory admissions change over time
- How well the predictive models perform
- Whether higher AQI tends to align with higher hospital burden
- Which AQI and time-based features are most important for prediction
"""))

    display(widgets.HBox([state_dropdown, model_dropdown, aqi_dropdown]))

    state_pred = pred_df[pred_df["state"] == state].copy()
    state_model = model_df[model_df["state"] == state].copy()

    # 1. Actual vs predicted over time
    plt.figure()
    plt.plot(state_pred["week"], state_pred["total_respiratory_admissions"], label="Actual")
    plt.plot(state_pred["week"], state_pred[model_col], label="Predicted")
    plt.title(f"{state}: Actual vs Predicted Respiratory Admissions")
    plt.xlabel("Week")
    plt.ylabel("Admissions")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # 2. AQI vs admissions scatter
    plt.figure()
    plt.scatter(state_model[aqi_metric], state_model["total_respiratory_admissions"])
    plt.title(f"{state}: {aqi_metric} vs Respiratory Admissions")
    plt.xlabel(aqi_metric)
    plt.ylabel("Total Respiratory Admissions")
    plt.tight_layout()
    plt.show()

    # 3. AQI over time
    plt.figure()
    plt.plot(state_model["week"], state_model[aqi_metric])
    plt.title(f"{state}: {aqi_metric} Over Time")
    plt.xlabel("Week")
    plt.ylabel("AQI")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # 4. Feature importance
    top_features = feature_importance.head(10)

    plt.figure()
    plt.barh(top_features["feature"], top_features["importance"])
    plt.gca().invert_yaxis()
    plt.title("Top 10 Random Forest Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    # 5. Short interpretation block
    display(Markdown(f"""
### Dashboard Interpretation for {state}
This view compares actual respiratory admissions with model predictions and shows how AQI conditions vary over time.
The scatter plot helps evaluate whether higher AQI is associated with increased respiratory admissions.
The feature importance chart highlights which AQI and temporal variables contribute most strongly to model predictions.
"""))

In [5]:
interactive_dashboard = widgets.interactive_output(
    update_dashboard,
    {
        "state": state_dropdown,
        "model_col": model_dropdown,
        "aqi_metric": aqi_dropdown
    }
)

display(widgets.HBox([state_dropdown, model_dropdown, aqi_dropdown]))
display(interactive_dashboard)

Output()